# Semana 9 — Projeto Prático de Data Warehouse e Modelagem Dimensional - SQL

Mini Data Warehouse de e-commerce usando o dataset Olist.

## Objetivos

- Diferenciar OLTP e OLAP
- Executar ETL com Pandas
- Criar dimensões e tabela fato
- Identificar PK, FK e Surrogate Key
- Criar Star Schema
- Consultar o modelo com SQL
- Interpretar métricas de BI

## Arquivos necessários

Baixe no Kaggle o dataset Brazilian E-Commerce Public Dataset by Olist e coloque na pasta do notebook:

- olist_orders_dataset.csv
- olist_order_items_dataset.csv
- olist_customers_dataset.csv
- olist_products_dataset.csv
- product_category_name_translation.csv

In [1]:
import pandas as pd
import duckdb
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)

# 1. Cenário de negócio

O diretor quer responder:

1. Qual foi o faturamento total?
2. Quais estados mais faturaram?
3. Quais categorias mais venderam?
4. Qual foi o ticket médio?
5. Como o faturamento evoluiu ao longo do tempo?

# 2. Extract — carregando fontes OLTP

In [2]:
pedidos = pd.read_csv(r'C:\Users\ferna\OneDrive\Área de Trabalho\SCTEC - PRATICA PT2\Semana_9\Dataset\olist_orders_dataset.csv')
itens = pd.read_csv(r'C:\Users\ferna\OneDrive\Área de Trabalho\SCTEC - PRATICA PT2\Semana_9\Dataset\olist_order_items_dataset.csv')
clientes = pd.read_csv(r'C:\Users\ferna\OneDrive\Área de Trabalho\SCTEC - PRATICA PT2\Semana_9\Dataset\olist_customers_dataset.csv')
produtos = pd.read_csv(r'C:\Users\ferna\OneDrive\Área de Trabalho\SCTEC - PRATICA PT2\Semana_9\Dataset\olist_products_dataset.csv')
traducao_categoria = pd.read_csv(r'C:\Users\ferna\OneDrive\Área de Trabalho\SCTEC - PRATICA PT2\Semana_9\Dataset\product_category_name_translation.csv')

print('orders:', pedidos.shape)
print('items:', itens.shape)
print('customers:', clientes.shape)
print('products:', produtos.shape)
print('category_translation:', traducao_categoria.shape)

orders: (99441, 8)
items: (112650, 7)
customers: (99441, 5)
products: (32951, 9)
category_translation: (71, 2)


## Exercício 1 🟢 Fácil — OLTP

Responda:

1. Por que essas tabelas podem ser consideradas fontes OLTP?
2. Qual tabela registra o pedido?
3. Qual tabela registra os itens de cada pedido?
4. Qual tabela descreve os clientes?
5. Qual tabela descreve os produtos?

# 3. Análise exploratória inicial

In [3]:
pedidos.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [4]:
itens.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [5]:
clientes.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [6]:
produtos.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


## Exercício 2 🟢 Fácil — Explorando os dados

Use `.info()`, `.shape` e `.isnull().sum()` para investigar as tabelas.

In [7]:
# Investigue a tabela orders
# SUA RESPOSTA AQUI

pedidos.info()
itens.info()
clientes.info()
produtos.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB
<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 no

In [8]:
# Investigue a tabela items
# SUA RESPOSTA AQUI

itens.isnull().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [9]:
# Investigue a tabela customers
# SUA RESPOSTA AQUI
clientes.isnull().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [10]:
# Investigue a tabela products
# SUA RESPOSTA AQUI
pedidos.isnull().sum()


order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

# 4. Transform — tratamento dos dados

Transformações comuns: converter tipos, remover duplicidades, tratar nulos, padronizar nomes e criar colunas derivadas.

In [11]:
pedidos['order_purchase_timestamp'] = pd.to_datetime(pedidos['order_purchase_timestamp'])
pedidos['order_approved_at'] = pd.to_datetime(pedidos['order_approved_at'])
pedidos['order_delivered_customer_date'] = pd.to_datetime(pedidos['order_delivered_customer_date'])
pedidos['order_estimated_delivery_date'] = pd.to_datetime(pedidos['order_estimated_delivery_date'])

pedidos[['order_purchase_timestamp', 'order_approved_at', 'order_delivered_customer_date']].head()

,order_purchase_timestamp,order_approved_at,order_delivered_customer_date
0,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-10 21:25:13
1,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-08-07 15:27:45
2,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-17 18:06:29
3,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-12-02 00:28:42
4,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-16 18:17:02


## Exercício 3 🟢 Fácil — Criando atributos de tempo

Crie as colunas `ano`, `mes`, `dia`, `trimestre` e `ano_mes` a partir de `order_purchase_timestamp`.

In [12]:
# SUA RESPOSTA AQUI

pedidos['order_purchase_timestamp'] = pd.to_datetime(pedidos['order_purchase_timestamp'])
pedidos['order_approved_at'] = pd.to_datetime(pedidos['order_approved_at'])
pedidos['order_delivered_customer_date'] = pd.to_datetime(pedidos['order_delivered_customer_date'])
pedidos['order_estimated_delivery_date'] = pd.to_datetime(pedidos['order_estimated_delivery_date'])
pedidos['ano'] = pedidos['order_purchase_timestamp'].dt.year
pedidos['mes'] = pedidos['order_purchase_timestamp'].dt.month
pedidos['dia'] = pedidos['order_purchase_timestamp'].dt.day
pedidos['trimestre'] = pedidos['order_purchase_timestamp'].dt.quarter
pedidos['ano_mes'] = pedidos['order_purchase_timestamp'].dt.to_period('M').astype(str)
pedidos[['order_purchase_timestamp', 'ano', 'mes', 'dia', 'trimestre', 'ano_mes']].head()

,order_purchase_timestamp,ano,mes,dia,trimestre,ano_mes
0,2017-10-02 10:56:33,2017,10,2,4,2017-10
1,2018-07-24 20:41:37,2018,7,24,3,2018-07
2,2018-08-08 08:38:49,2018,8,8,3,2018-08
3,2017-11-18 19:28:06,2017,11,18,4,2017-11
4,2018-02-13 21:18:39,2018,2,13,1,2018-02


## Exercício 4 🟢 Fácil — Remoção de duplicidades

Verifique se existem duplicidades nas tabelas principais e remova-as.

In [13]:

print('Duplicados orders:', pedidos.duplicated().sum())
print('Duplicados items:', itens.duplicated().sum())
print('Duplicados customers:', clientes.duplicated().sum())
print('Duplicados products:', produtos.duplicated().sum())
pedidos = pedidos.drop_duplicates()
itens = itens.drop_duplicates()
clientes = clientes.drop_duplicates()
produtos = produtos.drop_duplicates()


Duplicados orders: 0
Duplicados items: 0
Duplicados customers: 0
Duplicados products: 0


## Exercício 5 🟡 Médio — Filtro de pedidos válidos

Considere apenas pedidos com status `delivered` e crie `produtos_entregue`.

In [14]:
pedidos_delivered = pedidos[pedidos['order_status'] == 'delivered'].copy()
pedidos_delivered.shape

(96478, 13)

# 5. Modelagem dimensional

Modelo esperado:

```text
dim_cliente
     |
dim_produto -- fato_vendas -- dim_tempo
```

A tabela fato guarda eventos mensuráveis. As dimensões guardam contexto.

# 6. Criando a dimensão cliente

In [15]:
dim_cliente = clientes[['customer_id', 'customer_unique_id', 'customer_city', 'customer_state']].drop_duplicates().copy()
dim_cliente = dim_cliente.reset_index(drop=True)
dim_cliente['cliente_sk'] = dim_cliente.index + 1
dim_cliente = dim_cliente[['cliente_sk', 'customer_id', 'customer_unique_id', 'customer_city', 'customer_state']]
dim_cliente.head()

,cliente_sk,customer_id,customer_unique_id,customer_city,customer_state
0,1,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,franca,SP
1,2,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,sao bernardo do campo,SP
2,3,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,sao paulo,SP
3,4,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,mogi das cruzes,SP
4,5,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,campinas,SP


## Exercício 6 🟡 Médio — PK, SK e chave natural

1. Qual coluna é a Surrogate Key da dimensão cliente?
2. Qual coluna veio do sistema de origem?
3. Por que não usar CPF, e-mail ou código da origem como PK física do DW?

# 7. Criando a dimensão produto

In [16]:
dim_produto = produtos[['product_id', 'product_category_name']].drop_duplicates().copy()
dim_produto = dim_produto.merge(traducao_categoria, on='product_category_name', how='left')
dim_produto['product_category_name_english'] = dim_produto['product_category_name_english'].fillna('unknown')
dim_produto = dim_produto.reset_index(drop=True)
dim_produto['produto_sk'] = dim_produto.index + 1
dim_produto = dim_produto[['produto_sk', 'product_id', 'product_category_name', 'product_category_name_english']]
dim_produto.head()

,produto_sk,product_id,product_category_name,product_category_name_english
0,1,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery
1,2,3aa071139cb16b67ca9e5dea641aaa2f,artes,art
2,3,96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure
3,4,cef67bcfe19066a932b7673e239eb23d,bebes,baby
4,5,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares


## Exercício 7 🟡 Médio — Star Schema vs Snowflake

1. Esse modelo se parece mais com Star Schema ou Snowflake?
2. Como ficaria se categoria fosse uma tabela separada?
3. Qual opção costuma ser mais simples para Power BI?

# 8. Criando a dimensão tempo

In [17]:
#verifica se tem duplicidade de data e copia a tabela para criar a dimensão tempo

dim_tempo = pedidos_delivered[['order_purchase_timestamp']].drop_duplicates().copy()

### Crie as colunas necessárias para a dimensão tempo
dim_tempo['data'] = dim_tempo['order_purchase_timestamp'].dt.date #cria a coluna data apenas com a data, sem o horário
dim_tempo['ano'] = dim_tempo['order_purchase_timestamp'].dt.year #cria a coluna ano apenas com o ano da data
dim_tempo['mes'] = dim_tempo['order_purchase_timestamp'].dt.month #cria a coluna mes apenas com o mês da data
dim_tempo['dia'] = dim_tempo['order_purchase_timestamp'].dt.day #cria a coluna dia apenas com o dia da data
dim_tempo['trimestre'] = dim_tempo['order_purchase_timestamp'].dt.quarter #cria a coluna trimestre apenas com o trimestre da data


dim_tempo['ano_mes'] = dim_tempo['order_purchase_timestamp'].dt.to_period('M').astype(str) #cria a coluna ano_mes apenas com o ano e o mês da data, no formato "YYYY-MM"
dim_tempo = dim_tempo.drop_duplicates('data').reset_index(drop=True) #remove as linhas duplicadas com base na coluna data e reseta o índice
dim_tempo['tempo_sk'] = dim_tempo.index + 1 #cria a coluna tempo_sk com os valores sequenciais
dim_tempo = dim_tempo[['tempo_sk', 'data', 'ano', 'mes', 'dia', 'trimestre', 'ano_mes']] #reordena as colunas da dimensão tempo
dim_tempo.head() #exibe as primeiras linhas da dimensão tempo

,tempo_sk,data,ano,mes,dia,trimestre,ano_mes
0,1,2017-10-02,2017,10,2,4,2017-10
1,2,2018-07-24,2018,7,24,3,2018-07
2,3,2018-08-08,2018,8,8,3,2018-08
3,4,2017-11-18,2017,11,18,4,2017-11
4,5,2018-02-13,2018,2,13,1,2018-02


## Exercício 8 🟡 Médio — Dimensão tempo

1. Por que uma dimensão tempo é útil em BI?
2. Que análises ela permite?
3. Qual coluna é a SK da dimensão tempo?

# 9. Criando a tabela fato

Evento: um item vendido dentro de um pedido.

Granularidade: uma linha por item vendido em um pedido.

In [18]:
fato_vendas = itens[['order_id', 'order_item_id', 'product_id', 'price', 'freight_value']].copy()
fato_vendas = fato_vendas.merge(pedidos_delivered[['order_id', 'customer_id', 'order_purchase_timestamp']], on='order_id', how='inner')
fato_vendas.head()

,order_id,order_item_id,product_id,price,freight_value,customer_id,order_purchase_timestamp
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,58.90,13.29,3ce436f183e68e07877b285a838db11a,2017-09-13 08:59:02
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,2017-04-26 10:53:06
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,2018-01-14 14:33:31
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,2018-08-08 10:00:35
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,2017-02-04 13:57:51


## Exercício 9 🟡 Médio — Relacionando fato e dimensões

Complete a construção da `fato_vendas` trazendo `cliente_sk`, `produto_sk` e `tempo_sk`.

In [19]:
fato_vendas = fato_vendas.merge(dim_cliente[['cliente_sk', 'customer_id']], on='customer_id', how='left')
fato_vendas = fato_vendas.merge(dim_produto[['produto_sk', 'product_id']], on='product_id', how='left')
fato_vendas['data'] = fato_vendas['order_purchase_timestamp'].dt.date
fato_vendas = fato_vendas.merge(dim_tempo[['tempo_sk', 'data']], on='data', how='left')
fato_vendas.head()

,order_id,order_item_id,product_id,price,freight_value,customer_id,order_purchase_timestamp,cliente_sk,produto_sk,data,tempo_sk
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,58.90,13.29,3ce436f183e68e07877b285a838db11a,2017-09-13 08:59:02,65558,25866,2017-09-13,64
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,2017-04-26 10:53:06,34266,27231,2017-04-26,125
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,2018-01-14 14:33:31,34956,22625,2018-01-14,222
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,2018-08-08 10:00:35,51764,15404,2018-08-08,3
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,2017-02-04 13:57:51,7603,8863,2017-02-04,512


## Exercício 10 🟡 Médio — Métricas da fato

Crie `valor_produto`, `valor_frete` e `valor_total`.

In [20]:
fato_vendas['valor_produto'] = fato_vendas['price']
fato_vendas['valor_frete'] = fato_vendas['freight_value']
fato_vendas['valor_total'] = fato_vendas['valor_produto'] + fato_vendas['valor_frete']

## Exercício 11 🔴 Difícil — Fato final

Deixe a fato com: `order_id`, `order_item_id`, `cliente_sk`, `produto_sk`, `tempo_sk`, `valor_produto`, `valor_frete`, `valor_total`.

Depois, crie `venda_sk`.

In [21]:
fato_vendas_final = fato_vendas[['order_id','order_item_id','cliente_sk','produto_sk','tempo_sk','valor_produto','valor_frete','valor_total']].copy()
fato_vendas_final = fato_vendas_final.reset_index(drop=True)
fato_vendas_final['venda_sk'] = fato_vendas_final.index + 1
fato_vendas_final = fato_vendas_final[['venda_sk','order_id','order_item_id','cliente_sk','produto_sk','tempo_sk','valor_produto','valor_frete','valor_total']]
fato_vendas_final.head()

,venda_sk,order_id,order_item_id,cliente_sk,produto_sk,tempo_sk,valor_produto,valor_frete,valor_total
0,1,00010242fe8c5a6d1ba2dd792cb16214,1,65558,25866,64,58.90,13.29,72.19
1,2,00018f77f2f0320c557190d7a144bdd3,1,34266,27231,125,239.90,19.93,259.83
2,3,000229ec398224ef6ca0657da4fc703e,1,34956,22625,222,199.00,17.87,216.87
3,4,00024acbcdf0a6daa1e931b038114c75,1,51764,15404,3,12.99,12.79,25.78
4,5,00042b26cf59d7ce69dfabb4e55b4fd9,1,7603,8863,512,199.90,18.14,218.04


# 10. Validando o Star Schema

## Exercício 12 🔴 Difícil — Documentação do modelo


| Tabela | Tipo | PK | FK | Descrição |
|---|---|---|---|---|
| dim_cliente | Dimensão | cliente_sk | - | Contexto do cliente |
| dim_produto | Dimensão | produto_sk | - | Contexto do produto |
| dim_tempo | Dimensão | tempo_sk | - | Contexto temporal |
| fato_vendas | Fato | venda_sk | cliente_sk, produto_sk, tempo_sk | Evento mensurável de venda |

Granularidade: uma linha por item vendido por pedido.

# 11. SQL Analítico com DuckDB

In [22]:
con = duckdb.connect()
con.register('dim_cliente', dim_cliente) # registra a dimensão cliente no banco de dados
con.register('dim_produto', dim_produto) # registra a dimensão produto no banco de dados
con.register('dim_tempo', dim_tempo) # registra a dimensão tempo no banco de dados
con.register('fato_vendas', fato_vendas_final) # registra a tabela fato vendas no banco de dados

# Semana 9 — SQL para Data Warehouse

Nesta semana, vamos sair da etapa de construção do modelo dimensional e usar o Star Schema para responder perguntas de BI com SQL.

## Objetivos da Semana 9

Ao final das 3 aulas, você deverá conseguir:

- Usar `SELECT`, `WHERE` e `ORDER BY` em consultas analíticas.
- Fazer `INNER JOIN` entre tabela fato e dimensões.
- Usar `LEFT JOIN` para investigar ausência de relacionamento.
- Criar métricas com `GROUP BY`, `SUM`, `COUNT` e `AVG`.
- Filtrar agregações com `HAVING`.
- Usar funções de data como `DATE_TRUNC` e `EXTRACT`.

## Aula 1 da Semana 9 — SQL básico em DW

### Roteiro teórico

Em um Data Warehouse, o SQL é usado principalmente para análise.  
Diferente de um sistema transacional, onde o foco é inserir e atualizar registros, aqui o foco é consultar, agregar e transformar dados em indicadores.

A estrutura básica de uma consulta é:

```sql
SELECT colunas
FROM tabela
WHERE condição
ORDER BY coluna;
```

No nosso modelo:

- `fato_vendas` guarda os eventos mensuráveis de venda.
- `dim_cliente` explica quem comprou e onde.
- `dim_produto` explica o que foi vendido.
- `dim_tempo` permite analisar a venda no tempo.

## Consulta exemplo — Faturamento total

In [23]:
con.sql("""
SELECT SUM(valor_total) AS faturamento_total
FROM fato_vendas
""").df() # consulta para calcular o faturamento total da loja, somando a coluna valor_total da tabela fato_vendas

,faturamento_total
0,1.541977e+07


### Exemplo slide 5 - semana 9

In [24]:
con.sql("""
SELECT
    venda_sk,
    order_id,
    valor_produto,
    valor_frete,
    valor_total
FROM fato_vendas
LIMIT 20
""").df()

,venda_sk,order_id,valor_produto,valor_frete,valor_total
0,1,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19
1,2,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83
2,3,000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87
3,4,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78
4,5,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04
5,6,00048cc3ae777c65dbb7d2a0634bc1ea,21.90,12.69,34.59
6,7,00054e8431b9d7675808bcb819fb4a32,19.90,11.85,31.75
7,8,000576fe39319847cbb9d288c5617fa6,810.00,70.75,880.75
8,9,0005a1a1728c9d785b8e2b08b904576c,145.95,11.65,157.60
9,10,0005f50442cb953dcd1d21e1fb923495,53.99,11.40,65.39


### Exemplo slide 7 semana 9

In [25]:
con.sql("""SELECT
   venda_sk,
   order_id,
   valor_produto,
   valor_frete,
   valor_total
FROM fato_vendas
WHERE valor_total>500
ORDER BY valor_total DESC
LIMIT 20
""").df()

,venda_sk,order_id,valor_produto,valor_frete,valor_total
0,3482,0812eb902a67711a1cb742b3cdaa65ae,6735.00,194.31,6929.31
1,109785,fefacc66af859508bf1a7934eab1e97f,6729.00,193.21,6922.21
2,105496,f5136e38d1a14a4dbd87dff67da82701,6499.00,227.66,6726.66
3,72721,a96610ab360d42a2e5335a3998b4718a,4799.00,151.34,4950.34
4,10998,199af31afc78c699f0dbf71fb178d4d4,4690.00,74.34,4764.34
5,60712,8dbc85d1447242f3b127dda390d56e19,4590.00,91.78,4681.78
6,28537,426a9742b533fc6fed17d1fd6d143d7e,4399.87,113.45,4513.32
7,55421,80dfedb6d17bf23539beeef3c768f4d7,3999.00,195.76,4194.76
8,44822,68101694e5c5dc7330c91e1bbc36214f,4099.99,75.27,4175.26
9,76618,b239ca7cd485940b31882363b52e6674,4059.00,104.51,4163.51


### Exemplo — usando `WHERE` e `ORDER BY`

A consulta abaixo busca vendas com valor total maior que 200 e ordena da maior para a menor.

In [26]:
con.sql("""
SELECT
    venda_sk,
    order_id,
    valor_total
FROM fato_vendas
WHERE valor_total > 200
ORDER BY valor_total DESC
LIMIT 20
""").df()

,venda_sk,order_id,valor_total
0,3482,0812eb902a67711a1cb742b3cdaa65ae,6929.31
1,109785,fefacc66af859508bf1a7934eab1e97f,6922.21
2,105496,f5136e38d1a14a4dbd87dff67da82701,6726.66
3,72721,a96610ab360d42a2e5335a3998b4718a,4950.34
4,10998,199af31afc78c699f0dbf71fb178d4d4,4764.34
5,60712,8dbc85d1447242f3b127dda390d56e19,4681.78
6,28537,426a9742b533fc6fed17d1fd6d143d7e,4513.32
7,55421,80dfedb6d17bf23539beeef3c768f4d7,4194.76
8,44822,68101694e5c5dc7330c91e1bbc36214f,4175.26
9,76618,b239ca7cd485940b31882363b52e6674,4163.51


## Exercício 13 🟢 Fácil — SELECT em tabela fato

Liste as colunas `venda_sk`, `order_id`, `valor_produto`, `valor_frete` e `valor_total` das 15 primeiras vendas.

In [27]:
con.sql("""
SELECT
    venda_sk,
    order_id,
    valor_produto,
    valor_frete,
    valor_total
FROM fato_vendas
LIMIT 15
""").df()

,venda_sk,order_id,valor_produto,valor_frete,valor_total
0,1,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19
1,2,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83
2,3,000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87
3,4,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78
4,5,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04
5,6,00048cc3ae777c65dbb7d2a0634bc1ea,21.90,12.69,34.59
6,7,00054e8431b9d7675808bcb819fb4a32,19.90,11.85,31.75
7,8,000576fe39319847cbb9d288c5617fa6,810.00,70.75,880.75
8,9,0005a1a1728c9d785b8e2b08b904576c,145.95,11.65,157.60
9,10,0005f50442cb953dcd1d21e1fb923495,53.99,11.40,65.39


## Exercício 14 🟢 Fácil — Filtro com WHERE

Liste as vendas cujo `valor_frete` seja maior que 50.  
Mostre `venda_sk`, `order_id`, `valor_frete` e `valor_total`.

In [28]:
con.sql("""
SELECT
    venda_sk,
    order_id,
    valor_frete,
    valor_total
FROM fato_vendas
WHERE valor_frete > 50
LIMIT 10
""").df()

,venda_sk,order_id,valor_frete,valor_total
0,8,000576fe39319847cbb9d288c5617fa6,70.75,880.75
1,74,002b430ff89b3a24c31a1170acbbedea,65.56,265.55
2,83,0030ff924c38549807645976adeef2c0,67.24,292.24
3,97,0036757472ece3dde52fd4bfd929c90e,66.04,203.03
4,109,003edccf16bc5ec447f592913b3df2b4,50.85,64.85
5,123,00471463a6106056c1a2a809f70de640,85.97,265.96
6,128,004ca5ae248069d68e8594df8abf6ce0,61.18,221.08
7,155,00602f25bffa1dcfb71e202fbf9824fb,54.02,93.92
8,161,0066a1fdaee16ad5022c5ef979d0b661,87.28,226.28
9,185,0078a358a14592b887eb140ef515f5ab,82.86,336.38


## Exercício 15 🟡 Médio — Ordenação para ranking

Liste as 20 vendas de maior `valor_total`.  
Mostre `venda_sk`, `order_id`, `valor_produto`, `valor_frete` e `valor_total`.

In [29]:
con.sql("""
SELECT
    venda_sk,
    order_id,
    valor_produto,
    valor_frete,
    valor_total
FROM fato_vendas
ORDER BY valor_total DESC
LIMIT 20
""").df()

,venda_sk,order_id,valor_produto,valor_frete,valor_total
0,3482,0812eb902a67711a1cb742b3cdaa65ae,6735.00,194.31,6929.31
1,109785,fefacc66af859508bf1a7934eab1e97f,6729.00,193.21,6922.21
2,105496,f5136e38d1a14a4dbd87dff67da82701,6499.00,227.66,6726.66
3,72721,a96610ab360d42a2e5335a3998b4718a,4799.00,151.34,4950.34
4,10998,199af31afc78c699f0dbf71fb178d4d4,4690.00,74.34,4764.34
5,60712,8dbc85d1447242f3b127dda390d56e19,4590.00,91.78,4681.78
6,28537,426a9742b533fc6fed17d1fd6d143d7e,4399.87,113.45,4513.32
7,55421,80dfedb6d17bf23539beeef3c768f4d7,3999.00,195.76,4194.76
8,44822,68101694e5c5dc7330c91e1bbc36214f,4099.99,75.27,4175.26
9,76618,b239ca7cd485940b31882363b52e6674,4059.00,104.51,4163.51


### JOIN

Em um Star Schema, normalmente a análise começa na tabela fato e depois se conecta às dimensões.

Exemplo de raciocínio:

> Quero faturamento por estado.  
> O faturamento está na `fato_vendas`.  
> O estado está na `dim_cliente`.  
> Então preciso fazer JOIN entre fato e dimensão cliente.

Tipos importantes:

- `INNER JOIN`: retorna apenas registros que possuem correspondência nos dois lados.
- `LEFT JOIN`: mantém todos os registros da tabela da esquerda, mesmo sem correspondência na direita.

### Exemplo — fato + dimensão cliente

In [30]:
con.sql("""
SELECT
    f.venda_sk,
    f.order_id,
    f.valor_total,
    c.customer_state
FROM fato_vendas f
INNER JOIN dim_cliente c
    ON f.cliente_sk = c.cliente_sk
LIMIT 10
""").df()

,venda_sk,order_id,valor_total,customer_state
0,1,00010242fe8c5a6d1ba2dd792cb16214,72.19,RJ
1,2,00018f77f2f0320c557190d7a144bdd3,259.83,SP
2,3,000229ec398224ef6ca0657da4fc703e,216.87,MG
3,4,00024acbcdf0a6daa1e931b038114c75,25.78,SP
4,5,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,SP
5,6,00048cc3ae777c65dbb7d2a0634bc1ea,34.59,MG
6,7,00054e8431b9d7675808bcb819fb4a32,31.75,SP
7,8,000576fe39319847cbb9d288c5617fa6,880.75,SP
8,9,0005a1a1728c9d785b8e2b08b904576c,157.60,SP
9,10,0005f50442cb953dcd1d21e1fb923495,65.39,SP


### Exemplo — fato + dimensão produto + dimensão tempo

In [31]:
con.sql("""
SELECT
    f.venda_sk,
    f.order_id,
    p.product_category_name_english AS categoria,
    t.ano_mes,
    f.valor_total
FROM fato_vendas f
INNER JOIN dim_produto p
    ON f.produto_sk = p.produto_sk
INNER JOIN dim_tempo t
    ON f.tempo_sk = t.tempo_sk
LIMIT 20
""").df()

,venda_sk,order_id,categoria,ano_mes,valor_total
0,1,00010242fe8c5a6d1ba2dd792cb16214,cool_stuff,2017-09,72.19
1,2,00018f77f2f0320c557190d7a144bdd3,pet_shop,2017-04,259.83
2,3,000229ec398224ef6ca0657da4fc703e,furniture_decor,2018-01,216.87
3,4,00024acbcdf0a6daa1e931b038114c75,perfumery,2018-08,25.78
4,5,00042b26cf59d7ce69dfabb4e55b4fd9,garden_tools,2017-02,218.04
5,6,00048cc3ae777c65dbb7d2a0634bc1ea,housewares,2017-05,34.59
6,7,00054e8431b9d7675808bcb819fb4a32,telephony,2017-12,31.75
7,8,000576fe39319847cbb9d288c5617fa6,garden_tools,2018-07,880.75
8,9,0005a1a1728c9d785b8e2b08b904576c,health_beauty,2018-03,157.60
9,10,0005f50442cb953dcd1d21e1fb923495,books_technical,2018-07,65.39


#### Exemplo 2 (slide 11 - semana 9)

In [32]:
con.sql("""
SELECT
    v.venda_sk,
    v.order_id,
    v.cliente_sk,
    c.customer_city,
    c.customer_state,
    v.valor_total
 FROM fato_vendas v
 LEFT JOIN dim_cliente c
    ON v.cliente_sk = c.cliente_sk
 ORDER BY v.valor_total DESC
 LIMIT 20;
 """).df()

,venda_sk,order_id,cliente_sk,customer_city,customer_state,valor_total
0,3482,0812eb902a67711a1cb742b3cdaa65ae,13463,campo grande,MS,6929.31
1,109785,fefacc66af859508bf1a7934eab1e97f,21172,vitoria,ES,6922.21
2,105496,f5136e38d1a14a4dbd87dff67da82701,3907,marilia,SP,6726.66
3,72721,a96610ab360d42a2e5335a3998b4718a,27305,araruama,RJ,4950.34
4,10998,199af31afc78c699f0dbf71fb178d4d4,12062,maua,SP,4764.34
5,60712,8dbc85d1447242f3b127dda390d56e19,40678,joao pessoa,PB,4681.78
6,28537,426a9742b533fc6fed17d1fd6d143d7e,64496,sao paulo,SP,4513.32
7,55421,80dfedb6d17bf23539beeef3c768f4d7,98345,brasilia,DF,4194.76
8,44822,68101694e5c5dc7330c91e1bbc36214f,5899,bom jesus do galho,MG,4175.26
9,76618,b239ca7cd485940b31882363b52e6674,90197,nova lima,MG,4163.51


#### AULA 3 - AULA PRÁTICA

## Exercício 16 🟡 Médio — Faturamento por estado

In [ ]:
con.sql("""
-- SUA CONSULTA AQUI
""").df()

## Exercício 17 🟡 Médio — Top 10 categorias

In [ ]:
con.sql("""
-- escreva aqui seu código
""").df()

## Exercício 18 🔴 Difícil — Ticket médio por estado

In [ ]:
con.sql("""
-- SUA CONSULTA SQL AQUI
""").df()

## Exercício 19 🔴 Difícil — Evolução mensal de faturamento

In [ ]:
con.sql("""
-- SUA CONSULTA SQL AQUI
""").df()

In [ ]:
faturamento_mensal = con.sql("""
SELECT t.ano_mes, SUM(f.valor_total) AS faturamento_total
FROM fato_vendas f
JOIN dim_tempo t ON f.tempo_sk = t.tempo_sk
GROUP BY t.ano_mes
ORDER BY t.ano_mes
""").df()
faturamento_mensal.head()

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(faturamento_mensal['ano_mes'], faturamento_mensal['faturamento_total'])
plt.xticks(rotation=90)
plt.title('Evolução Mensal do Faturamento')
plt.xlabel('Ano-Mês')
plt.ylabel('Faturamento')
plt.show()